In [ ]:
import requests
from datetime import datetime
from langchain_core.documents import Document

# ✅ 전체 학사일정 데이터를 LangChain 문서로 변환
def load_haksa_documents():
    url = "https://www.scnu.ac.kr/haksa/sv/schdulView/selectSvList.do"
    response = requests.post(url, data={"sysId": "SCNU"})
    data = response.json()  # ⭐ 전체 학사일정 (2018~현재)

    docs = []
    for item in data:
        start = item.get("bgnde","")
        end = item.get("endde","")
        title = item.get("schdulTitle", "")
        description = item.get("schdulCn", "")
        
        start_fmt = datetime.strptime(start, "%Y/%m/%d").strftime("%Y년 %m월 %d일")
        end_fmt = datetime.strptime(end, "%Y/%m/%d").strftime("%Y년 %m월 %d일")
        
        if start == end:
            content = f"{title} 일정은 {start_fmt}입니다."
        else:
            content = f"{title} 일정은 {start_fmt}부터 {end_fmt}까지입니다."
        
        docs.append(Document(
            page_content=content,
            metadata={
                "start_date": start,
                "end_date": end,
                "description": description,
                "all_day": item.get("alldayAt", "")
            }
        ))

    return docs

docs = load_haksa_documents()
print(f"총 문서 수: {len(docs)}개\n")

# 앞에 10개만 출력
for i, doc in enumerate(docs[:10]):
    print(f"[{i+1}] {doc.page_content}")
    print(doc.metadata)
    print("-" * 50)

일정 테스트 일정은 2018년 08월 03일입니다.
{'start_date': '2018/08/03', 'end_date': '2018/08/03', 'description': '일정 테스트 내용', 'all_day': 'Y'}
